# Глава 9. Алгоритмическая оптимизация
[анонс главы]

## Мотивация
В IV главе мы отмечали, что при всех плюсах у Трансформерной архитектуры есть недостатки, серьёзно усложняющие расчет.

Если модель настроена в режим энкодинга, самовнимание должно считать все попарные взаимодействия (каждого токен с каждым), его вычислительная сложность и требуемая память растёт как $O(L^2)$, где $L$ - длина последовательности. Пока контекст короткий, это мало заметно, но на больших контекстах, таких как длинные документы, большие репозитории кода, многошаговые диалоги, квадратичная сложность становится существенным ограничением. Энкодинг везде, где он нужен (например, обучение модели, префилл перед генерацией, классификация текста) становится очень дорогим.

В режиме инференса токены порождаются по одному, каждый новый токен требует отдельного прохода модели и также возникает квадратичная сложность. Частично процесс можно ускорить за счёт использования KV кэша (см. описание в главе IV), но и его нужно читать из медленной памяти (HBM), из-за чего тензорные ядра GPU простаивают - просто ждут данные большую часть времени. При том, что размер KV-кэша тоже линейно растёт с длиной контекста.

Желание ускорить этот процесс породило целое направление исследований. В этой главе рассмотрим три важные идеи. Первая идея - удешевить расчет самовнимания, заменив полное вычисление на приближенное, это можно назвать разреженным самовниманием. Вторая идея - оптимизировать само вычисление, перераспределив его компоненты, самый яркий пример это метод FlashAttention. Третья - ускорить расчет за счёт перераспредления задач. Подход назывют спекулятивным декодированием.

## Приближенное внимание

### Sparse Transformer
В 2019 году [(Child et al.)](https://arxiv.org/abs/1904.10509) из OpenAI пошли вероятно по самому простому пути - решили ограничить множество токенов, которые участвуют в подсчете внимания. Благо, для этого в Трансформере уже есть готовый механизм, который можно использовать "из коробки", и называется он *маска внимания*. Модель назвали Sparse Transformer. Модель использовала сразу два окна внимания. Локальный - всегда смотрим на ближайший конектст. Strided - смотрим на каждый k-ый токен.

Описываемая логика одинаково применима как для энкодинга, так и для декодинга. Единственное отличие - в режиме авторегресионной генерации (GPT) дополнительно умножаем матрицу внимания на нижнетреугольная маска.

### Longformer
[[Beltagi et al, 2020]](https://arxiv.org/abs/2004.05150) продолжили ту же логику, но скорректировали набор используемых масок. Во-первых, добавили ещё один паттерн - глобальное внимание, который безусловно включают важные токены, например те которые расположены в самом начале. А во-вторых, регулярную сетку (strided mask) заменили на скользяее окно с прореживанием, в рамках которого оставляется только каждый k-ый токен (dilated mask).

Если зафиксируем общее кол-во активных токенов, то с подобным разреживанием доступный контекст можно серьезно расширить. По этой причине модель назвали Longformer = Long Document Transformer. 

Иллюстрация масок ниже. 

<img src="img/longformer.png" width=500>

### BigBird
[[Zaheer, 2021]](https://arxiv.org/abs/2007.14062) решили, что набор масок важно расширить масками со случайными токенами.

<img src="img/bigbird.png" width=500>

Теоретическое обоснование для этого они взяли из теории графов. Графами «Small World» (графами высокой связности) называют графы, где путь между любой парой вершин короткий. Добавление случайных рёбер в граф довольно быстро превращают малосвязный граф в сильно связванный (как бы добавляется возможность "телепортации" из вершины в вершину). Этот прицип является основой многих социальных эффектов, в частности теории шести рукопожатий.

Важно праивильно проинтерпертировать. На уровне одного слоя внимания какой-то особенной связности нет: запрос стандартно агрегирует сигнал от доступных ему ключей $k$. Но, если мы рассматриваем всю глубину модели - суперкпозицию множества слоев внимания, то видим, как сигнал может распространяться от токена к токену, в том числе через добавленные случайные связи. Авторы показали, что такая схема приближает выразительность полного внимания, при этом оставаясь линейной по времени вычисления.

<img src="img/routing.png" width=450>

### Adaptive Attention Span
Разные головы внимания могут отвечать за разные зависимости, например, какие-то могут быть локальными, какие-то долгосрочными. В этом случае значительная часть вычислений оказывается ненужной. В 2019 году [(Sukhbaatar et al)](https://arxiv.org/abs/1905.07799) в Facebook AI Research предложили давать возможность блоку самовнимания самому определять, на какие токены он смотрит. Метод назвали __Adaptive Attention Span__. Для каждой головы самовнимания вводится собственный обучаемый параметр $z$, определяющий реальный span: $z\in[0,S]$. Вместо жёсткого маскирования доступных позиций решили использовать плавную функцию $m_z(x)$, которая плавно растёт от 0 до 1. Логарифм переводит её в штраф:

$$\alpha_{ij} = \operatorname{softmax}_j \left( \frac{q_i k_j^T}{\sqrt d} + \log m_z(i-j) \right)$$

Дополнительно авторы вводят регуляризацию длины span, чтобы модель была мотивирована не использовать контекст без необходимости $\mathcal L=\mathcal L_{\text{LM}}+\lambda\sum_h z_h$.

### Reformer
[[Kitaev et al, 2020]](https://arxiv.org/abs/2001.04451) из Google обратили внимание, что сам расчет внимания в Трансформерной архитектуре избыточен и предложили свою архитектуру, которую назвали Reformer (Reversable Transformer). В ней они реализовали три модификации.

Во-первых, они заметили что на практике внимание распределяется неравномерно, обычно оно достается всего нескольким токенам из контекста, а значение остальных оказывается около нуля. Соотвественно нас в первую очередь интересуют токены дающие максимальное произведение, а остальными можно пренебречь. Максимум скалряного произведения достигается, когда q и k сонаправлены, иными словами, чем ближе векторные представления k и q, тем больше связь между ними. Если сегментировать вектора и распредлеить их в однородные "корзины", тогда перебор можно ограничить рамками одной корзины.

Для этого они прибегли к идеально подходящему под эту задачу алгоритму приближенного поиска соседей LSH (locality-sensitive hashing). Идея в том, что все ветктора описываются небольшим набором случайными проекцияй, которые можно очень быстро посчитать и не тратить время на полный расчет расстояния между векторами. Тогда с большой вероятностью близкие или идентичные представления попадут в одним проекций. Останется перебрать пары из одной корзины.

<img src="img/reformer.png" width=600>

Во-вторых, обучение сети методом обратного распролстранения ошибки требует хранения активаций всех слоев, поэтому стандартно на прямом проходе оптимизатор их запоминает в памяти и на обратном проходе достает. Но если хочется сэкономить на памяти, на обратном проходе активации можно не запомнианть, а вычилсять повторно - это занимает больше времени, но меньше расходуется память. В случае с классическим Трансофрмером это невозможно, так как . Поэтому авторы предложили переконфигурировать остаточные связи - . Теперь тут нет рекурсии и взоды можно восстановить. Назали это обратимыми слоями (reversible residuals).

Третья новация связана с вычислением полносвязного слоя (FFN). Это вычисление использует самый большой тензор, что дает пиковое использование памяти. Но поскольку применяется независимо ко всем токенам входной последовательности, вычисление можно разбить на последовательные блоки. Общее кол-во не меняется, но пиковое испольование памяти уменьшается.

Статья дала старт целому направлению "Efficient Transformers", но мейнстримом модель так и не стала, уступив пальму первенства методам типа FlashAttention (о нем ниже).

### Routing Transformer
[[Roy et al, 2020]](https://arxiv.org/abs/2003.05997) вышли с похожей на Reformer идеей - сделать маску разреженности динамической, зависящей от входа, но вместо LSH они использовали кластеризацию. Модель назвали __Routing Transformer__, показывая, что. В качестве метода кластеризации использовали k-means (онлайн его вариант: примеры показываются один за другим, кластеризация уточняется). Каждый запрос $q$ считает внимание только с ключами $k$ своего же кластера и таким образом сокращается общее кол-во вычислений.

### Linformer
Для экономии вычислений [[Wang et al, 2021]](https://arxiv.org/abs/2006.04768) идут чуть по другому пути, они не разделяют последовательность токенов на сегменты, а плотно упаковывают всю эту последовательность представлений в небольшой фиксированной размерности вектор псевдотокенов. Делается это для представлений K и V через два добавленных перед вниманием слоя проекций, после чего вычисляется стандартный механизм внимания. В батчевом режиме он уже не квадратный, а прямоугольный $QK_{dense}^T$, то есть каждый оригинальный токен $q$ агрегирует сигнал от сжатого набора псевдотокенов $k$.

Модель назвали __Linformer__ акцентируя внимание на том, что из-за фикисрованной размерности последовательности псевдотокенов, сложность вычисления становится линейной: ведь независимо от размера входного контекста, нам всегда достаточно сравниться с k псевдотокенами.

<img src="img/linformer1.png" width=500>

### Performer
Команда [[Choromankspiy et al, 2022]](https://arxiv.org/abs/2009.14794) из Google предложили заменить саму формулу расчета внимания на ее приближенный вариант, основываясь на паре математически приемов, главным из которых было разложении ядровой функции (kernel trick). Модель назвали __Performer__, а алгоритм приближенного внимания FAVOR+ (Fast Attention Via positive Orthogonal Random features). В результате упрощения формулы в ней появилась ассоциативность и стало возможным умножать на матрицу V раньше, чем умножать на матрицу Q. Тем самым пропала необхоимость вообще материализовать квадратичную матрицу $QK^T$.

Напомним, что обычное внимание вычисляется как $\mathrm{Att}(Q,K,V)=D^{-1}AV$, где $A_{ij}=\exp(q_i^\top k_j/\sqrt d)$ и $D=\mathrm{diag}(A\mathbf 1_L)$. 

Построение матрицы $A$ размера $L\times L$, имеет квадратичную по длине контекста $L$ сложность. Авторы $A_{ij}=\mathcal K(q_i,k_j)$ предлагают представление $\mathcal K(q,k)=\mathbb E_{\omega}[\varphi_\omega(q)^\top\varphi_\omega(k)]$ через отображение в признаковое пространство. Если построить явное приближение $\varphi:\mathbb R^d\to\mathbb R^m$, то $A\approx\varphi(Q)\varphi(K)^\top$, и матрица факторизуется на два «узких» множителя.

Основная сложность в том, что не очень понятно, как выбирать конкретное отображение $\varphi$ в пространство фичей. По классике для разложения RBF ядра (экспоненты) используют тригонометрические случайные признаки:

$$\varphi_{\mathrm{trig}}(x) = \frac{1}{\sqrt{m}}\Big[\cos(\omega_1^\top x),\dots,\cos(\omega_m^\top x),\; \sin(\omega_1^\top x),\dots,\sin(\omega_m^\top x)\Big], \qquad \omega_i \stackrel{\text{iid}}{\sim} \mathcal{N}(0,I_d)$$

Но проблема с ними в том, что они дают знакопеременные координаты, из-за чего сложно приближать малые значения ядра — а именно такие значения доминируют в реальных матрицах внимания. Поэтому авторы предложили использовать другое представление со строго положительными признаками:

$$\varphi(x)=\frac{\exp(-\|x\|^2/2)}{\sqrt m}\Big[\exp(\omega_1^\top x),\dots,\exp(\omega_m^\top x)\Big],\qquad \omega_i\sim\mathcal N(0,I_d),$$

для которого $\mathbb E[\varphi(q)^\top\varphi(k)]=\exp(q^\top k)$ точно, а дисперсия стремится к нулю там же, где стремится к нулю само ядро. 

Дополнительно векторы $\omega_i$ генерируются попарно ортогональными, что строго уменьшает дисперсию оценки.

После замены $QK^T$ на приближенное представление становится возможной перестановка скобок в произведении, где вместо $(\varphi(Q)\varphi(K)^\top)V$ мы можем посчитать $\varphi(Q)\,(\varphi(K)^\top V)$. Знаменатель находится аналогично: $\hat D=\mathrm{diag}\big(\varphi(Q)(\varphi(K)^\top\mathbf 1_L)\big)$. То есть не материализовать проблемную квадратную матрицу, в просто считать проивзедение двух векторов. В авторегрессионном режиме внимание вообще обновляется за константное время $O(1)$, а длина последовательности нигде не зашита в веса, что делает подход гибким.

$$\mathrm{out}_i=\hat d_i^{-1}\,\varphi(q_i)^\top S_i,\qquad S_i=S_{i-1}+\varphi(k_i)v_i^\top\in\mathbb R^{m\times d},$$

На практике замена не очень окупилась: Performer уступил полному вниманию, поскольку приемлемая дисперсия требует довольно большой размерности $m$. Тем не менее важный пример того, что линейное внимание может быть выведено из аппроксимационных соображений. Похожий прием испольщовался в более поздней модели MAMBA (подробнее в следующей главе).

### Blockwise Transformer / BlockBERT
В Facebook AI предложили __BlockBERT__. Идея в том, что контекст длины $L$ разбивается на блоки одинакового размера и каждый блок сравнивается с одним случайным блоком. Связка реализуется случайной перестановкой $\pi$. Вместо того чтобы каждый токен каждого блока смотрел на все $n$ токенов, внимание разрешается только между определёнными парами блоков.

Например, токены блока $1$ могут связываться с токенами блока $2$, блок $2$ с блоком $3$ и т. д. Разные головы самовнимания и разные слои Трансформера используют разные перемстановки. За счёт этого информация постепенно распространяется дальше. Ещё один плюс в том, что блочно-разреженные матрицы лучше обрабатываются на видеокартах, чем произвольную разреженную матрицу.

### BP Траснформер
В методе Binary-Partitioning Transformer развивает идею, давайте представлять конекст не блоками, а бинарным деревом. Тогда с одной стороны можем ограничить вычисление внимания k соседними блоками, а рассматривая разные уровни детализации, покрыть весь контекст.

[уточнить алгоритм]


$$O\left(k n\log\frac{n}{k}\right) \quad \text{вместо} \quad O(n^2)$$




## Экстраполяция конекста
### Интерпоялция позиций
### NTK-scaling
### YARN

## Масштабирование самовнимания

### MQA
[[Shazeer, 2019]](https://arxiv.org/abs/1911.02150) предложили сыграть на избыточности внимания и оставить только по одному экзмепляру проекций для K и V векторов. Разнообращие обьеспечивается только для запросов Q, поскольку считает, наиболее важный компонент сигнала, для Q оригинальное количество представлений. Таким образом общее количество вычислений не меняется, но использование памяти под KV кэш заметно сокращается, а значит и генерация заметно ускоряется. Подход назвали __Multi-Query Attention__. За оптимизацию приходится платить некоторой просадкой качества и меньшей стабильностью обучения, поскольку ограничивается разнообразие представлений K и V.

<img src="img/mqa.webp" width=400>

### GQA
[[Ainslie et al, 2023]](https://arxiv.org/abs/2305.13245) предложили __Grouped-Query Attention__ компромисс между полным вниманием (MHA) и MQA. Можество запросов делятся на G групп, и каждая группа разделяет одну голову K/V. существующую MHA-модель можно дёшево «дообучить» (uptrain) в GQA. В том числе по этой причине GQA стал стандартом в моделях LLaMA-2/3, Mistral и других.

<img src="img/gqa.png" width=400>

### MLA
[[Liu et al, 2024]](https://arxiv.org/abs/2405.04434) __Multi-head Latent Attention__ (DeepSeek-V2) меняет сам вопрос. Вместо «как поделить меньшее число голов K/V» спрашивается «зачем вообще хранить полноразмерные K и V». K и V совместно сжимаются низкоранговой проекцией в маленький латентный вектор, и в кэше лежит только он; при вычислении внимания пер-головые K и V восстанавливаются обратной проекцией на лету. В DeepSeek-V2 это дало сокращение KV-кэша примерно на 93% относительно MHA, причём, по их измерениям, не ценой качества, а с небольшим выигрышем. 

Платой стала сложность: низкоранговое сжатие плохо дружит с RoPE, поэтому введён «расщеплённый» (decoupled) RoPE — отдельная небольшая часть размерностей несёт позиционную информацию; и приём «поглощения весов» (weight absorption), сворачивающий проекции, чтобы восстановление не стоило лишних вычислений. MLA архитектурно более инвазивен, чем GQA, но и потенциально мощнее: он торгует дополнительными вычислениями (распаковка) за резкое снижение памяти и трафика

<img src="img/mla.png" width=500>

### NSA
Команда [(Yuan et al, 2025)](https://arxiv.org/abs/2502.11089) из DeepSeek вернулись к старой идее разреженности со своим вариантом внимания __NSA (Native Sparse Attention)__. Они сформулировали пайплайн, состоящий из трех парадлельных экстракторов. Во-первых, это сжатый поиск: блоки токенов суммаризируются в компактные представлений. Во-вторых, точный поиск: выбираются самые релевантные блоки токенов, к которым затем применяется полное внимание. В третьих используется скользящее окно (свежий локальный контекст). Выходы всех трех компонентов смешиваются обучаемым гейтом. 

Два принципиальных отличия от методов 2020 года. Во-первых, NSA обучаема нативно — разреженность присутствует с самого предобучения, а не плявляется в первый раз на инференсе. Во-вторых, она аппаратно-согласована: шаблон спроектирован под GPU (сбалансированная арифметическая интенсивность, блочные ядра, выровненные под группировку GQA), поэтому теоретическая экономия операций превращается в реальное ускорение по времени. На последовательностях в 64k NSA заметно быстрее полного внимания на декодировании, прямом и обратном проходах, при этом не уступая ему в качестве.

<img src="img/nsa.png" width=500>

## FlashAttention
(Dao et al, 2022) задались вопросом, можно ли перекомпоновать алгоритм расчета, оставив его точным, но ускорить вычисление. Оказалось, что не только можно, но это еще и очень эффективно. Так родился алгоритм __FlashAttention__ модификация обычного самовнимания, ускоряющая расчет. Ключевое наблюдение — стандартное внимание упирается не в вычисления, а в память: оно записывает в медленную HBM огромную промежуточную матрицу n×n и читает её обратно. Значит, надо минимизировать обращения к HBM, а не число операций.

### FlashAttention-1
[[Dao et al, 2022]](https://arxiv.org/abs/2205.14135)<Br>
FlashAttention компонует вычисление внимания таким образом, что становится более оптимальным по I/O. На GPU есть быстрая SRAM память и медленная HBM, хочется больше вычислений делать на SRAM, 

Attention матрица нарезается на куски (процесс называется тайлинг): блоки Q, K, V подгружаются из HBM в быструю память SRAM и обрабатываются по частям. 
Softmax считается «онлайн» по частям (с бегущими максимумом и суммой), поэтому полная матрица внимания нигде не материализуется целиком. 
На обратном проходе используется пересчёт: вместо хранения большой матрицы её восстанавливают из компактной статистики. 

В итоге память линейна по n, число обращений к HBM резко падает, а итоговое ускорение по времени — в 2–4 раза, почти бесплатно и с сохранением точности.

<img src="img/flash1.png" width=600>

### FlashAttention-2
[[Dao et al, 2023]](https://arxiv.org/abs/2307.08691)<br>
__FlashAttention-2__, вторая версия (2023), не меняет алгоритм, но переписывает распараллеливание. Сокращается доля «не-matmul» операций (тензорные ядра GPU считают матричное умножение во много раз быстрее, чем специальный блок, отвечающий за экспоненту в softmax), вычисление параллелится вдоль длины последовательности, а работа лучше распределяется между варпами и блоками, уменьшая трафик через разделяемую память. Это даёт ещё около двукратного ускорения и доводит утилизацию до примерно 50–70% на A100. На H100, однако, версия достигала лишь ~35%, потому что не использовала особенности нового железа

### FlashAttention-3
В 2024 году [[Shah et al, 2024]](https://arxiv.org/abs/2407.08608) выпустили __FlashAttention-3__, третью версия алгоритма. В этот раз целью была максимально возможная синхронизацию с железом, а конкренто с архитектурой Nvidia Hopper (H100). Авторы при этом не отходят от своего главного принципа - внимание по-прежнему остается точным.

Три приёма: использование асинхронности (warp-specialization — одни варпы через TMA асинхронно подгружают данные, другие в это время считают на тензорных ядрах WGMMA, перекрывая память и вычисления); чередование (ping-pong) блочного matmul и softmax, чтобы медленная экспонента считалась одновременно с матричным умножением; и низкая точность FP8 с блочным квантованием и «incoherent processing», которые удерживают точность (примерно в 2,6 раза меньше ошибка, чем у наивного FP8). 

В результате простой на видеокартах H100 сокращается с 65% до 25%, а скорость генерации становится в 1.5–2 раза выше второй версии.

Алгоритм FlashAttention совершил своего рода революцию в вычислениях языковых моделей на видеокартах и фактически обесценил целое направление приближённых методов,  и показав, что того же эффекта можно добиться с сохранением точности и сделав их бесполезными.



# Спекулятивное декодирование
[(Leviaithan et al, 2023)](https://arxiv.org/pdf/2211.17192) описали идею спекулятивного декодирования (speculative decoding), основой которой является наблюдение, что генерация обычно неравномерна по сложности. Какие-то куски ответа сгенерировать просто (```2 x 2 = ```), для каких-то нужно серьезное рассуждение. Почему бы не переключаться между разными моделями на ходу? Допустим есть большая языковая модель (целевая, target), и есть компактная языковая модель (черновая, draft). Дадим малой модели возможность быстро генерировать продолжение на несколько шагов, а затем большая модель проверит качество ее продолжения. Если оно удовлетворительное, то идем дальше, если неудовлетворительное, целевая модель перегенерирует то же самое продолжение, но уже самостоятельно.

Шаг верификации продолжения дешевый, поскольку происходит за одну итерацию Траснформера может проверяться сразу много нагенерированных черновых токенов. В роли модели-черновика часто выбирают более базовую версия той же целевой модели. Приемка вероятностная: целевая модель сравнивает две вероятности сегенерированного продолжения, свою $P(t)$ и дочерней модели $Q(t)$. Если дочерняя модель переоценила вероятность токена $(Q > P)$, продолжение принимается с вероятностью $P/Q$. Если недооценила, заменяем на вариант целевой модели.

Таким образом, выигрыш есть, если черновик дёшев, а доля принятых токенов высока.

### Blockwise Parallel Decoding
В 2018 году еще до появления самого термина "спеулятивное декодирование" описали модель Blockwise Parallel Decoding. Идея простая - к выходу последнего скрытого состояния модели добавляется несколько лёгких «голов» (heads), каждая из которых независимо предсказывает токен на позиции +1, +2, +3 и так далее.

### Medusa
В области генеративных языковых моделей существует целый набор методов, озаглавленный неавторегрессионная генерация (NAT), основная идея которого - генерация продолжения не из одного, а сразу из сразу нескольких токенов. 
Первая модель была вообще без верификации . Позже появилась модель Blockwise Parallel Decoding. 

[[Cai et al, 2024]](https://arxiv.org/abs/2401.10774) взяли старую модель BPD и предложили пару модификаций процесса верификации. Главная доработка - вместо генерации одного продолжения генерируется сразу множество и организуется в виде дерева. Так появилась модель __Medusa__.  В первой версии модели Medusa-1 достаточно обучить только головы, остальные веса замораживается, что обсепечивает высокую скорость работы. Во второй версии модель обучается целиком.

Как генерируется дерево продолжений? Каждая голова генерирует top-k токенов и по этим наборам строится декартово произведение всех возможных комбинаций, на основании которой строится дерево. Далее все комбинации укладываются в одну плоскую структуру - это сделано, чтобы можно было посчитать вероятности каждого токена за одну итерацию. А чтобы гарантировать, что токен видит только свой префикс используется маска внимания. В MEDUSA такую маску внимания называют Tree Attention. И затем остается верифицировать все сгенерированные продолжения, выбрав наиболее вероятное.

<img src="img/medusa1.png" width=300>

В работе описано три механизма верификации. В рамках "жадной" стратегии, мы просто на каждом шаге выбираем токен с наибольшей вероятностью по целевой модели. В рамках стратегии "Typical Acceptance" мы считаем вероятность каждого токена по основной модели и сравниваем ее с порогом уверенности, скорректированным на энтропию его распределния (при малой энтропии порог уменьшается). В рамках стратегии `nucleus` для каждой позиции в дереве алгоритм берёт распределение вероятностей всех токенов, полученное от основной модели, сортирует их по убыванию и начинает добавлять токены в так называемое «ядро», пока суммарная вероятность накопленных токенов не достигнет порога `top_p` (обычно 0.8). Токен-кандидат от головы MEDUSA принимается, если он попал в это ядро, и отклоняется в противном случае.

Явный минус подхода MEDUSA в том, что каждая голова генерирует токен независимо от других (находятся вне локального контекста, особенно последние токены), поэтому точность генерации явно падает.

### Hydra
[(Ankner, 2024)](https://arxiv.org/abs/2402.05109) решили исправить эту независимость и сделали генерирующие головы (heads) последовательно зависимыми: каждая получает на вход токены, предложенные предыдущими. По сути черновик превращается из набора независимых предсказанных токенов в нормальную последовательную модель, что заметно повышает среднюю длину принятого фрагмента.

<img src="img/hydra.png" width=300>

### EAGLE
[[Li et al, 2024]](https://arxiv.org/abs/2401.15077) ключевая мысль первой версии модели __EAGLE-1__ (Extrapolation Algorithm for Greater Language-model Efficiency): генерировать следующий токен не по токеном, но и скртые преставение  предпоследнего скрытого состояния. Идея в том, что непрерывное скрытое представление гораздо лучше в качестве сигнала, чем дискретные токены. затем из предсказанного признака получают токен через готовую LM-голову целевой модели. Скрытое состояние передается вместе со сгенерированным токеном.

Черновик делает фиксированное число авторегрессионных шагов, обычно 4-8. Затем верификатор (полнаая модель) оценивает и оставляет наиболее длинную принятую последовательность. Черновая модель представляет сильно усеченную версию основной модели: слой эмбединов и выходной HEAD слой берутся из оригинальной модели, а между ними добавляется один траснформеный слой. Модель в авторегрессионном режиме генерирует продолжение и . Для верификации заимствуется идея из MEDUSA, генерируется сразу дерево возможных продолжений, которые оцениваются механикой TreeAttetnion.

<img src="img/eagle1_1.png" width=500>

[(Li et al, 2024)](https://arxiv.org/abs/2406.16858) во второй версии модели __EAGLE-2__ корректировку решили делать не после генерации чернового дерева, а в процессе его построения с помощью beam search. Черновая модель сама оценивает уверенность в ответе.

[[Li et al, 2025]](https://arxiv.org/abs/2503.01840) в версии __EAGLE-3__ решили отказаться от предсказания скрытого состояния, а предсказывать сразу токен. Вместо одних только верхних признаков используется конкатенация признаков с разных слоев. Кроме того, многошаговый процесс черновика симулируется уже на обучении, устраняя рассинхрон между обучением и инференсом. Это даёт порядка 3–6,5× ускорения относительно обычной генерации и на 20–40% выше EAGLE-2.


# Обзор методов

## Черновик без модели
В работе __SpecDec__ (Xia et al., 2022) впервые вводят сам термин «speculative decoding». Для задач, где ожидается, что выход будет сильно похож на вход (исправление грамматики, постредактирование). Черновиком служит непосредственно входной текст: модель проверяет, совпадает ли её собственный выход с входом, и продолжает копировать, пока совпадение держится. Это первый пример того, что позже разовьётся в целое семейство методов копирования (LLMA, Prompt Lookup).

###  Копирование из входа
Во многих сценариях выход модели заведомо похож на вход. Например, при исправлении грамматики или точечном редактировании или при retrieval-augmented генерации, когда модель цитирует найденные документы. В этих случаях в качестве черновика можно использовать скопированные куски входного текста. Так устроен в частности __Aggressive Decoding__ (Sun et al., 2021) — исторически первый метод этого семейства. Принцип работы в том, что модель ищет во входном промпте последовательность из последних N сгенерированных токенов, и если находит, то просто копирует следующие K токенов в выход.

Если модель ищет продолжения не во входном промпте, а в подгруженных документах, то это метод __LLMA__ (Yang et al., Microsoft, 2023). А если модель ищет и в промпте, и в документах, то это уже __Prompt Lookup Decoding__ (Saxena, 2023). В vLLM этот метод присутствует под именем `ngram`.

###  Поиск в хранилище
Возможность предыдущих методов сильно ограничены - если выход не повторяет вход, то копировать нечего. Но можно искать не в контексте, а в отдельном корпусе текстов. Так поступают [(He et al., 2023)](https://arxiv.org/abs/2311.08252) в методе **REST** (Retrieval-Based Speculative Decoding). Корпус текстов хранится в суффиксном массиве, суффикс текущей генерации работает как запрос, все найденные продолжения складываются в префиксное дерево, оттуда отбираются самые частотные ветви — и получившееся дерево кандидатов целевая модель проверяет за один проход. 

Такой подход сильно зависит от домена. Собранный на коде, он бесполезен для диалога. От этой зависимости можно избавиться, если наполнять хранилище он-лайн во время генерации. Суффиксы берутся из промптов и прошлых выходов самой модели. Так устроен **Suffix Decoding** [(Oliaro et al., Snowflake, 2025)](https://arxiv.org/abs/2411.04975): хранилище подстраивается под текущую нагрузку само. Длина и форма черновика выбираются по статистике накопленного дерева. Особенно заметно в агентных сценариях, где однотипные вызовы инструментов повторяются десятки раз. В vLLM доступен как `suffix`.

Логично попытаться объединить оба источника — статический корпус и растущую историю. Это сделали в **SAM-Decoding** [(Hu et al., 2025)](https://arxiv.org/abs/2411.10666). Заодно заменили суффиксное дерево суффиксным автоматом: поиск становится амортизированно константным вместо логарифмического. Идея та же, выигрыш чисто инженерный, но он важен: при коротком черновике и частых шагах накладные расходы на поиск сами начинают съедать экономию.

### Переиспользование собственных вычислений
На каждом шаге генерации модель выдаёт распределение по всему словарю, но физичсеки используется из него один токен. Остальные кандидаты из top-k пропадают. **Token Recycling** [(Luo et al., 2024)](https://arxiv.org/abs/2408.08696) сохраняет их в матрице «токен → вероятные продолжения» и на следующих шагах собирает из неё дерево черновика.

Похожий принцип применим и к отходам верификации. Токены, отвергнутые на предыдущих шагах, **Ouroboros** [(Zhao et al., 2024)](https://arxiv.org/abs/2402.13720) складывает в пул фраз-кандидатов и удлиняет этим пулом черновик, порождённый обычной маленькой моделью. Здесь сопоставление строк работает в спайке с нейросетевым драфтером.



### Черновик без модели

Драфтер не обязан быть нейросетью. Если продолжение можно угадать сопоставлением строк — найти в тексте место, где такой же фрагмент уже встречался, и взять то, что шло за ним, — черновик достаётся почти даром: не проход по сети, а поиск в структуре данных. Верификация остаётся прежней: целевая модель проверяет кандидатов за один параллельный проход.

Меняется экономика. Раз черновик бесплатен, его можно брать длинным и не бояться промахов. Но принятие становится всё-или-ничего: либо совпал целый кусок в десятки токенов, либо не совпало ничего. Там, где выход пересекается с доступным текстом, это даёт 2–3×; там, где не пересекается, — ровно ничего. Методы различаются тем, откуда берутся n-граммы для сопоставления.

**Поиск в хранилище.** Ограничение предыдущей группы очевидно: если выход не повторяет вход, копировать нечего. Следующий шаг — искать не в контексте, а в отдельном корпусе текстов. Так поступает **REST** [(He et al., 2023)](https://arxiv.org/abs/2311.08252): корпус хранится в суффиксном массиве, суффикс текущей генерации работает как запрос, все найденные продолжения складываются в префиксное дерево, оттуда отбираются самые частотные ветви — и получившееся дерево кандидатов целевая модель проверяет за один проход. Всё упирается в корпус: собранный на коде, он бесполезен для диалога.

От этой зависимости можно избавиться, если наполнять хранилище во время работы — из промптов и прошлых выходов самой системы. Так устроен **Suffix Decoding** [(Oliaro et al., Snowflake, 2025)](https://arxiv.org/abs/2411.04975): хранилище подстраивается под текущую нагрузку само, а длина и форма черновика выбираются по статистике накопленного дерева. Особенно заметно в агентных сценариях, где однотипные вызовы инструментов повторяются десятки раз. В vLLM доступен как `suffix`.

Оба источника — статический корпус и растущую историю — объединяет **SAM-Decoding** [(Hu et al., 2025)](https://arxiv.org/abs/2411.10666), заодно заменяя суффиксное дерево суффиксным автоматом: поиск становится амортизированно константным вместо логарифмического. Идея та же, выигрыш чисто инженерный, но он важен: при коротком черновике и частых шагах накладные расходы на поиск сами начинают съедать экономию.

**Переиспользование собственных вычислений.** Третий источник кандидатов — то, что модель уже посчитала и выбросила. На каждом шаге она выдаёт распределение по всему словарю, а используется из него один токен; остальные кандидаты из top-k пропадают. **Token Recycling** [(Luo et al., 2024)](https://arxiv.org/abs/2408.08696) сохраняет их в матрице «токен → вероятные продолжения» и на следующих шагах собирает из неё дерево черновика. По цене источник почти идеален: ничего дополнительно считать не нужно.

Тот же принцип применим и к отходам верификации: токены, отвергнутые на предыдущих шагах, **Ouroboros** [(Zhao et al., 2024)](https://arxiv.org/abs/2402.13720) складывает в пул фраз-кандидатов и удлиняет этим пулом черновик, порождённый обычной маленькой моделью. Метод стоит на границе семейства — сопоставление строк работает здесь не вместо нейросетевого драфтера, а поверх него.

### Самоспекулятивное декодирование
Часто в роли драфтера используют сокращённую версию орнигинальной модели. Как правило это та же модель, но с меньшим количеством слоев, а полная модель используется для верификации. Такой подход называется Self-Speculative Decoding. Из плюсов: поскольку модель по сути одна, можно переиспользовать её словарь и KV-кэш. Из минусов: уменьшение количества слоев (в глубину) всегда даёт более скромную экономию, чем уменьшение внутренней размерности (в ширину), обычно это всего 1.3–2x. Кроме того, выключение части слоев ломает исходную модель и нужно придумать какую-то адаптацию.

Проще всего взять в роли драфтера первые слои Трансформерной модели (например, 8 из 32). По такому пути, пошли [(Elhoushi et al., 2024)](https://arxiv.org/abs/2404.16710) в своем методе __LayerSkip__. В течение K итераций токен за токеном генерируется черновик продолжения, а затем прогоняется за один проход через полную модель для верификации. Чтобы гарантировать, что признаки генерируются более равномерно по слоям модели, модель заранее обучают со случайным выкидыванием слоев (Layer Dropout), а также с лоссом, который считается не на последнем, а на всех выходах сети $\mathcal{L} = \mathcal{L}_1 + ... \mathcal{L}_N$.

В методе __Kangaroo__ (2024) от предобучения отказались, но чтобы сгладить разницу между 8 и 32 слоями, добавили ещё один дополнительный слой, и предобучается только он. Кроме того, сделали драфтинг динамичнее, разрешили выходить при низкой уверенности.

Не обязательно всегда оставлять только первые слои, возможно произвольные комбинации слоев дадут результат лучше. Так решили сделать [(Zhang et al., 2023)](https://arxiv.org/abs/2309.08168) в методе __Draft & Verify__. Какие слои выключать подбирается один раз байесовской оптимизацией на небольшой выборке. Кроме того, они также ввели динамический критерий окончания драфтинга (модель сама говорит, проверяй). Если уверенность (вероятность наиболее вероятного токена) падает, то генерация черновика прекращается досрочно.

Если сделать отбор слоев динамическим, менчбщимся на каждый запрос, то получится метод __SWIFT__ от [(Xia et al., 2024)](https://arxiv.org/abs/2410.06916), где набор пропускаемых слоёв определяется прямо во время инференса, по ходу обработки конкретного запроса, а не подбирается заранее. Обучение не нужно вообще и есть адаптивность к домену, например, оптимальное подмножество слоёв для кода и для диалога различается.

### Мультитокенная генерация
Можно сэкономить на количестве проходов. Научить модель выдавать черновик из нескольких будущих токенов за один проход.

__PaSS: Parallel Speculative Sampling__ [(Monea et al., 2023)](https://arxiv.org/abs/2311.13581). К входу добавляются специальные обучаемые look-ahead эмбеддинги, играющие роль «плейсхолдеров» под будущие токены. За один проход модель заполняет их, порождая черновик.

В __Speculative Streaming__ от (Apple, 2024) предсказывается сразу несколько токенов, но вместо отдельных голов внутри одной и той же модели поддерживается несколько «потоков», каждый из которых отвечает за свою будущую позицию, и они обмениваются информацией через механизм внимания. Число дополнительных параметров на порядки меньше — что критично для развёртывания на устройствах.

__Multi-Token Prediction__ [(Gloeckle et al., 2024)](https://arxiv.org/abs/2404.19737). Модель предобучается с несколькими выходными головами, предсказывающими токены на несколько позиций вперёд. Изначально это делалось ради повышения качества, (считалось, что такая задача оказывается лучшим обучающим сигналом), но её побочным эффектом оказался готовый драфтер: на инференсе те же головы порождают черновик. Не дообучение адаптера поверх замороженной модели, а полноценное совместное предобучение.

Похожая идея реализована в Medusa, но там головы (heads) для генерации токенов навешены снаружи основной модели и в целом угадывают хуже, чем MTP. У MTP же способность вшита в сами представления, поэтому доля принятых токенов заметно выше. Самый известный пример MTP сегодня — модель DeepSeek-V3, где MTP-модуль стоит в предобучении, а на инференсе работает драфтером.

### Декодирование системой уравнений
Метод __Lookahead Decoding__ (декодирование с подсматриванием) от [(Fu et al., 2024)](https://arxiv.org/abs/2402.02057) использует известный математический трюк, известный как метод Якоби. Он испольуется для приближенного итеративного решения системы уравнений $Ax=b$. Идея в том, что берется начальное приближение $x_0 = (x^1_0 ... x^n_0)$ и каждое неизвестное выражается через остальные. Затем процесс повторяется до тех пор, пока $x_t$ не сойдется к решению $x$.

В нашем случае выбор токена зависит от выбора $k$ предыдущих, поэтому можем оформить эти зависимости системой из k уравнений и приближенно решить её. Метод сознательно тратит вычисления: он загружает простаивающие вычислительные блоки GPU, обменивая FLOPs на скорость ответа, поэтому метод хорош при малом батче и бесполезен при большом, когда GPU уже загружен.

Сходится это медленно, поэтому метод Lookahead попутно кэширует n-граммы, возникающие по ходу итераций, и может их переиспользовать при генерации другого черновика. Метод __CLLM: Consistency LLM__ от [(Kou et al.,  2024)](https://arxiv.org/abs/2403.00835) пытается ускорить медленное схождение методом Якоби за счет предобучения самой модели по набору траекторий Якоби, которые преварительно строятся. Цель - чтобы из произвольной начальной точки она за одну итерацию Якоби сходилась сразу целевой точке.


### Внешний драфтер
Часто в роли драфтера выступает самостоятельная маленькая модель — обычно это модель той же архитектуры, с тем же токенизатором, но на 1-2 порядка меньше по количеству параметров. Слишком малый драфтер будет генерировать плохие продолжения и часто отвергаться большой моделью, а слишком большой нивелировать выигрыш по скорости.

У нас цель повторить поведение большой модели, но если драфтер обучается отдельно, то он оптимизирует свою точность, а не похожесть на большую модель. Идея - давайте дистиллировать драфтер из выходов целевой модели. Так делают, например, [(Zhou et al., ICLR 2024)](https://arxiv.org/abs/2310.08461) в методе __DistillSpec__. В своей работе они показали, что, во-первых, это действительно важно, продемонстрировав ускорение 10-45% по сравнению со стандартным спекулятивным декодированием. Во-вторых, обозначили важность обучения именно на текстах, сгенерированных самим драфтером, а не на внешнем датасете. В-третьих, на обучении драфтера оптимизируется дивергенция между двумя распределениями токенов (драфтера и целевой модели) и выбор оптимальной метрики дивергенции сильно зависит от задачи и режима генерации.

Не обязательно обучать драфтер заранее, __Online Speculative Decoding__ [(Liu et al., 2023)](https://arxiv.org/abs/2310.07177) делает это прямо по мере генерации в продакшене. Данные возникают бесплатно: каждый отвергнутый токен — готовая пара «вход — правильный ответ», потому что целевая модель его уже выдала. Смысл в адаптации драфтера к разным доменам.

Можем заметить, что драфтер занимается тем же, что и большая модель - генерирует продолжение текста. И когда окно генерации большое, он сам начинает занимать много времени. Тогда почему бы не завести драфтер для драфтера? Эту идею реализовали в методе __Staged Speculative Decoding__ [(Spector & Ré, 2023)](https://arxiv.org/abs/2308.04623). Заодно они реструктурировали черновик из линейной последовательности (жадной генерации) в дерево (топ-k генерации), чтобы увеличить вероятность принятия. Механизм верификации дерева за один проход (Tree Attention) уже был известен в индустрии, он описан в методе SpecInfer.

В методе __Cascade Speculative Drafting__ [(Chen et al., 2023)](https://arxiv.org/abs/2312.11462) эту идею обобщили до пирамиды драфтеров, которые образуют *вертикальный каскад*: большую модель драфтит средняя, среднюю — маленькая. На дне обычная n-граммная модель. В спекулятивном декодировании вероятность принятия последовательности из $i$ токенов падает экспоненциально, поэтому тратить на дальние позиции столько же, сколько на ближние, невыгодно. Поэтому завели ещё *горизонтальный каскад*, который распределяет бюджет по позициям внутри черновика: ранние позиции получают дорогого драфтера, поздние — дешёвого.


Какой формы генериррвать набор кандидатов и по какому правилу большая модель решает, что из него принять.

### Дерево кандидатов

Когда мы генерируем цепочкой, если большая модель отвергла токен на позиции $i$, всё, что после него, выбрасывается, даже если оно было бы верным при другом $i$-м токене. Дерево решает эту проблему: на каждой позиции драфтер оставляет несколько кандидатов и продолжает каждый.

В работе __SpecInfer__ [(Miao et al., ASPLOS 2024)](https://arxiv.org/abs/2305.09781) впервые стали использовать дерево кандидатов. 
Авторы предлагают два варианта его построения. Первый (сложный) - они обучают ансамбль драфтеров (по принципу бустинга), далее все модели ансмабля генерируют токены и результаты агрегируются - если токен совпал, продолжается цепочка, если разошлись, то цепочка ветвится в дерево. Второй способ (простой) - ветвление по фиксированному расписанию.

Ключевой технический приём — проверка всего дерева за один проход большой модели. Ветки укладываются в одну последовательность, а маска внимания (__Tree attention__) устроена так, что каждый узел видит только своих предков, но не соседние ветки. Этот приём прижился и сегодня его используют в методах типа Medusa, Hydra, EAGLE и т.д.

```
        A B C D E
A       ✓
B       ✓ ✓
C       ✓   ✓
D       ✓ ✓   ✓
E       ✓ ✓     ✓
```

Форму дерева можно подбирать через оптимизацию, как это делают в __Sequoia__ [(Chen et al., 2024)](https://arxiv.org/abs/2402.12374). У можно оценить вероятность принятия. Форма дерева (сколько узлов и как они распределены по глубине и ветвям) выбирается так, чтобы максимизировать ожидаемое число принятых токенов за единицу времени. 

Кроме того предложили кандидаты на позиции выбирать сэмплированием без возвращения, а не просто топ-$k$. Из-за этого метод не теряет эффективность при высокой температуре, тогда как фиксированные деревья при ней заметно проседают.

__SpecExec__ [(Svirschevski et al., 2024)](https://arxiv.org/abs/2406.02532) рассчитан на другую экономику. Если веса большой модели не помещаются в память GPU и лежат в оперативной памяти или на диске, каждый её проход требует перекачки весов через шину и стоит очень дорого, а стоимость драфтера на этом фоне пренебрежима. Тогда разумно нагрузить каждый проход по максимуму: SpecExec строит деревья в сотни узлов, что в обычной постановке было бы расточительством. Пример показывает, что оптимальный размер черновика — не константа, а функция отношения стоимости прохода большой модели к стоимости драфтера.

### Правила приемки
Классическая приёмка работает так: на позиции $i$ токен черновика принимается с вероятностью $\min(1, p_i/q_i)$, где $p$ и $q$ — вероятности этого токена у большой модели и у драфтера. При первом отказе проверка останавливается. Интерпретация простая, если драфтер генерирует токен реже, чем нужно (P > Q), значит этого токена нужно больше и он прнимается безусловно. Если чаще, чем нужно (P > Q), то количество нужно уменьшить и мы прореживаем выдачу, берем только каждый $Q/P$. Но есть и другие способы верификации.

__SpecTr__ [(Sun et al., NeurIPS 2023)](https://arxiv.org/abs/2310.15141) даёт общую рамку для случая, когда на одну позицию есть несколько кандидатов (от нескольких черновиков или из дерева). Задача ставится как оптимальный транспорт: нужно так «перенести» вероятность с распределения драфтера на распределение большой модели, чтобы как можно чаще итоговый токен оказывался среди кандидатов. Из этой постановки получается алгоритм отбора, сохраняющий распределение большой модели, и верхняя оценка ускорения, которое вообще достижимо при данном числе кандидатов.

Можно оценивать при верификации вероятность не отдельных токенов, а вероятность совместного распределения всей последовательности. Так делают в __Block Verification__ [(Sun et al., 2024)](https://arxiv.org/abs/2403.10444). Идея в том что в этом случае мы не отсекаем ситуации, когда токен получает заниженную вероятность, а следующий за ним компенсирует это наоборот высокой вероятностью. Более того, авторы доказывают, что такая стратегия даёт максимальное ожидаемое число принятых токенов среди всех правил, сохраняющих распределение.

Ту же идею можно перенести на дерево кандидатов, как это делают, в __Traversal Verification__ [(Weng et al., 2025)](https://arxiv.org/abs/2505.12398). Если обычная проверка дерева идёт от корня к листьям и обрывает ветку на первом отвергнутом узле, то здесь обход идёт от листьев к корню, и решение принимается о целых путях.

## Декодирование с потерями
До этого момента мы предполагали, что распределение токенов пары (драфтер + большая модель) должно в точности совпадать с распределением токенов большой модели. Такие методы будем условно назвать "точными". Но за такую гарантию приходится жертвовать скоростью генерации, ведь большая модель обязана проверять каждый токен, а критерий приёмки достаточно строгий и отвергает все продолжения, в чём модели расходятся. Можно ли отказаться от части требований и ускорить генерацию? Можно. Такие методы будем называть спекулятивным декодированием "с потерями".

Например, можно смягчить критерий и принимать всё, что большая модель считает «достаточно вероятным». Эта идея появлялась ещё в первой работе по спекулятивному декодированию (Leviathan et al.) под названием *lenience* - там вероятности большой модели умножаются на коэффициент $\ell \ge 1$, и приём становится снисходительнее. В методе DistillSpec она исследована как отдельный режим. А в **Medusa** [(Cai et al., 2024)](https://arxiv.org/abs/2401.10774) похожий приём называется *Typical Acceptance*: токен принимается, если его вероятность по оценке большой модели превышает порог, который зависит от энтропии распределения — чем неопределённее сама большая модель на этой позиции, тем ниже порог и тем больше кандидатов проходит.

Аргумент авторов такой, что в реальном сэмплировании (с температурой, с выборкой по top-k) пользователь и так уже довольно сильно отклонился от исходного распрделения, так зачем так строго следовать критерию приёма, теряя время на отказах.

[(Kim et al., 2023)](https://arxiv.org/abs/2302.07863) дают драфтеру больше свободы и делают длину генерации гибкой. В своем методе __BiLD__ (Big Little Decoder) они позволили драфтеру генерировать текст, пока он уверен в своих предсказаниях, то есть пока вероятность наиболее вероятного токена выше порога $\alpha$. Как только появляется токен, в котором драфтер не вверен, он передаёт управление большой модели — это называют *fallback*. 

В этот момент большая модель проверяет всё, что драфтер написал до этого и сравнивает с своими вер-тями. Если на каких-то позициях разность оказалась больше порога $\beta$, то модель идет на самую раннюю такую позицию и ставит там свой токен  — это *rollback*. А если таковой не нашлось, то идет сразу на последнюю позицию (где драфтер не уверен) и ставит свой токен там. В обоих случаях после замены токена она передает управление обратно драфтеру и тот продолжает генерацию с этого места.

В классической схеме процедура приёмки не просто отсеивает плохие токены, она корректирует распределение так чтобы выход совпадал с целевым распределением при любом качестве драфтера. Здесь эта механика заменена на простое сравнение с порогом — оно ничего не компенсирует, поэтому драфтер начинает вносить заметный вклад в итоговое распределение, и совместное распрделение уходит от целевого.

[(Bachmann et al., 2025)](https://arxiv.org/abs/2501.19309) совсем отходят от принципа "всегда повторяй за большой моделью" и ставят во главу угла качество генерации. В методе **Judge Decoding** добавляется небольшой бинарный классификатор поверх скрытого представления большой модели, обученный на размеченных людьми примерах черновиков - здесь токен уместен, здесь нет. Именно этот классификатор решает, принимается ли данный токен.


### Ускорение на железе

**Основной механизм.** Спекулятивное декодирование выигрывает потому, что авторегрессионная генерация упирается в пропускную способность памяти, а не в вычисления: GPU простаивает, перекачивая веса. Верификация нескольких токенов сразу использует этот простой. Но с ростом размера батча система становится вычислительно-ограниченной, простоя больше нет, и спекуляция превращается в чистые накладные расходы. В замерах на vLLM ускорение EAGLE падает примерно с 1.96× при батче 1 до примерно 1.21× при батче 128, поэтому в продакшене спекуляцию обычно автоматически отключают выше некоторого порога загрузки.

**SmartSpec и TurboSpec.** Идея состоит в том, чтобы выбирать длину спекуляции динамически, исходя из оценки goodput при текущей загрузке сервера, а не фиксировать её конфигом. Отличие от всех алгоритмических работ — целевая функция: оптимизируется не ускорение одного запроса, а пропускная способность системы при соблюдении SLO.

**PEARL (2024).** Метод устраняет взаимное ожидание драфтера и таргета, вводя предварительную верификацию первого токена черновика параллельно с драфтингом остальных. Отличие от классической схемы в том, что там драфтер и таргет работают строго по очереди, и один из них всегда простаивает.

**TriForce и MagicDec (2024).** При длинном контексте узким местом становятся не веса, а KV-кэш, поэтому черновик здесь строится на разреженном или усечённом KV самой целевой модели. MagicDec приносит контринтуитивный результат: при достаточно длинном контексте спекулятивное декодирование выгодно **и** при больших батчах, поскольку бутылочное горлышко сместилось.

**Spec-Bench (Xia et al., 2024).** Стандартный бенчмарк для сравнения методов на одном железе и одних данных. Упомянуть его стоит как методологическую опору: заявленные в разных статьях ускорения между собой несопоставимы, потому что различаются модели, длины, оборудование и режим декодирования.
